# 01 — Prima lettura di NSL-KDD



**Prima di iniziare:** eseguire il download dalla root come descritto nel [README](../README.md) e selezionare il kernel **Python (NSL-KDD)**. Le [fonti dei dati](../docs/DATA_SOURCE.md) e il [manifest](../docs/DATA_MANIFEST.json) identificano la copia utilizzata.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Funziona avviando il kernel dalla root oppure dalla cartella notebooks.
project_root = Path.cwd()
if not (project_root / 'src' / 'data.py').is_file():
    project_root = project_root.parent
if not (project_root / 'src' / 'data.py').is_file():
    raise RuntimeError('Aprire il notebook dalla root del progetto o da notebooks/.')
sys.path.insert(0, str(project_root))

from src.data import CATEGORICAL_COLUMNS, FEATURE_COLUMNS, load_nsl_kdd

pd.set_option('display.max_columns', 12)
pd.set_option('display.max_rows', 45)

## 1. Leggere il file senza perdere la prima riga

Il CSV non ha un'intestazione: la prima riga è già un record. Il loader assegna i nomi delle 43 colonne e verifica la larghezza di ogni riga. La copia `.csv` ha lo stesso contenuto del TXT di origine.

In [ ]:
train_path = project_root / 'data' / 'raw' / 'KDDTrain+.csv'
train = load_nsl_kdd(train_path)

print(f'Record train: {train.shape[0]:,}')
print(f'Colonne totali: {train.shape[1]}')
print(f'Feature originali: {len(FEATURE_COLUMNS)}; poi attack_label e difficulty')

## 2. Guardare alcune connessioni

`head()` mostra i primi cinque record. Le colonne centrali vengono abbreviate nella vista per mantenere l'output leggibile; sono tutte presenti nel DataFrame.

In [ ]:
train.head()

## 3. Capire i tipi delle colonne

`info()` elenca tutte le colonne, i tipi e i conteggi non nulli. È una prima lettura: il controllo qualità completo arriverà nel ticket 4.

In [ ]:
train.info()

In [ ]:
train.dtypes.astype(str).value_counts().rename('numero_colonne').to_frame()

Le tre feature testuali sono **protocol_type**, **service** e **flag**. Le altre 38 feature sono memorizzate come numeri: ciò non implica che siano tutte quantità continue; per esempio alcuni interi rappresentano indicatori. **attack_label** è l'etichetta grezza, **difficulty** un metadato numerico. Entrambi saranno esclusi dagli input dei modelli.

In [ ]:
column_roles = pd.DataFrame([
    {'gruppo': 'Feature testuali', 'numero': len(CATEGORICAL_COLUMNS),
     'esempi': ', '.join(CATEGORICAL_COLUMNS)},
    {'gruppo': 'Feature numeriche', 'numero': len(FEATURE_COLUMNS) - len(CATEGORICAL_COLUMNS),
     'esempi': 'duration, src_bytes, count, serror_rate, ...'},
    {'gruppo': 'Target grezzo', 'numero': 1, 'esempi': 'attack_label'},
    {'gruppo': 'Metadato', 'numero': 1, 'esempi': 'difficulty'},
])
column_roles 

## 4. Riassumere le feature numeriche

`describe()` riporta conteggio, media, deviazione standard, minimo, quartili e massimo. Qui selezioniamo solo le 38 feature numeriche, lasciando fuori target e difficulty. La trasposizione mette una feature per riga. I valori vengono arrotondati soltanto per la visualizzazione.

In [ ]:
numeric_features = train.loc[:, list(FEATURE_COLUMNS)].select_dtypes(include='number')
numeric_features.describe().round(2).T

## 5. Riassumere le colonne testuali

Per stringhe e categorie, `describe(include='all')` mostra conteggio, numero di valori distinti, valore più frequente e relativa frequenza. `attack_label` compare qui solo per comprenderne la struttura: non stiamo ancora mappando gli attacchi nelle cinque famiglie.

In [ ]:
text_columns = [*CATEGORICAL_COLUMNS, 'attack_label']
train.loc[:, text_columns].describe(include='all').T

## 6. Legenda delle colonne principali

Ogni riga descrive una connessione. **Sorgente** indica chi avvia lo scambio, **destinazione** chi lo riceve. Questa legenda aiuta a leggere i nomi: non è una graduatoria dell'importanza delle feature per un modello.

### Connessione e attività dell'utente

| Colonna | Significato a parole |
|---|---|
| `duration` | Durata, in secondi. |
| `protocol_type` | Protocollo: per esempio TCP, UDP o ICMP. |
| `service` | Servizio di destinazione, per esempio HTTP o FTP. |
| `flag` | Stato della connessione: legenda delle sigle qui sotto. |
| `src_bytes` / `dst_bytes` | Byte di dati inviati / ricevuti dalla sorgente. |
| `land` | 1 se sorgente e destinazione coincidono per host e porta. |
| `wrong_fragment` | Numero di frammenti errati. |
| `urgent` | Numero di pacchetti urgenti. |
| `logged_in` | 1 se il login è riuscito. |
| `num_failed_logins` | Tentativi di login falliti. |
| `root_shell` | 1 se è stata ottenuta una shell con privilegi root. |
| `num_compromised` | Conteggio delle condizioni di compromissione rilevate. |
| `hot` | Conteggio di indicatori sensibili nel contenuto. |

Per `hot` e `num_compromised`, la scheda descrive contatori senza elencare completamente gli eventi: non attribuiamo loro una definizione più precisa del documento.

Fonti: [Lee, Stolfo e Mok, descrizione dei record e delle caratteristiche di base, §§2–3](https://ids.cs.columbia.edu/sites/default/files/wenke-kdd99.pdf); [indicatori del contenuto nella scheda KDD, UCI](https://kdd.ics.uci.edu/databases/kddcup99/task.html).


### Che cosa significa `flag`?

`flag` riassume lo **stato osservato della connessione**; non è il singolo bit SYN/ACK di un pacchetto TCP. Le sigle si leggono secondo le convenzioni Bro/Zeek documentate per `conn_state`.

| Valore | Lettura pratica |
|---|---|
| `SF` | Connessione aperta e terminata normalmente. |
| `S0` | Tentativo senza risposta osservata. |
| `REJ` | Tentativo rifiutato. |
| `S1` | Connessione stabilita, non terminata. |
| `S2` | Chiusura avviata dalla sorgente, senza risposta osservata. |
| `S3` | Chiusura avviata dalla destinazione, senza risposta osservata. |
| `RSTO` | Connessione stabilita, poi reset dalla sorgente. |
| `RSTR` | Reset inviato dalla destinazione. |
| `RSTOS0` | SYN e poi reset dalla sorgente, senza SYN-ACK osservato. |
| `SH` | SYN e poi FIN dalla sorgente, senza SYN-ACK osservato. |
| `OTH` | Traffico osservato senza il SYN iniziale. |

**SYN** avvia l'apertura, **FIN** segnala la chiusura, **RST** un'interruzione/reset TCP. Per esempio, `S0` indica che manca una risposta nella traccia osservata, non dimostra da solo un attacco. Anche `SF` non garantisce che il contenuto sia innocuo: `flag` e `attack_label` rispondono a domande diverse.

Fonte per le sigle: [documentazione ufficiale Zeek, `Conn::Info.conn_state`](https://docs.zeek.org/en/current/scripts/base/protocols/conn/main.zeek.html#type-Conn::Info). È un riferimento interpretativo delle sigle, non una verifica dell'estrattore storico usato per ciascun record.


### Contatori e quote: `count`, `srv_*`, `*_rate`, `dst_host_*`

| Colonna o famiglia | Significato a parole |
|---|---|
| `count` / `srv_count` | Connessioni allo stesso host / servizio negli ultimi due secondi. |
| `serror_rate` / `rerror_rate` | Quote di errori SYN / REJ nel gruppo dello stesso host. |
| `srv_serror_rate` / `srv_rerror_rate` | Le stesse quote, nel gruppo dello stesso servizio. |
| `same_srv_rate` / `diff_srv_rate` | Quote verso servizi uguali / diversi, nel gruppo dello stesso host. |
| `dst_host_*` | Statistiche per destinazione, su una finestra di connessioni anziché due secondi. |

Lettura pratica dei suffissi: `count` è un conteggio; `rate` una quota. Nei dati, una quota di **0,80 equivale all'80%**, non allo 0,80%. Un errore di connessione e un attacco non sono sinonimi.

Il prefisso cambia il gruppo su cui viene calcolata una statistica: per esempio `serror_rate` e `srv_serror_rate` descrivono lo stesso tipo di errore, ma non lo stesso insieme di connessioni.

Fonte: [gruppi temporali e per host nella specifica KDD, UCI](https://kdd.ics.uci.edu/databases/kddcup99/task.html). I [nomi completi delle colonne](https://kdd.ics.uci.edu/databases/kddcup99/kddcup.names) sono definiti nello schema originale.


### Etichetta e difficoltà: non sono feature da dare al modello

| Colonna | Significato e utilizzo |
|---|---|
| `attack_label` | Risposta corretta del dataset: `normal` oppure il nome dell'attacco. In seguito ricondurremo questi nomi alle cinque famiglie. |
| `difficulty` | Deriva da quanti dei 21 classificatori usati nella costruzione di NSL-KDD riconoscevano correttamente il record. Valori alti indicano record più facili per quei classificatori; non attacchi più gravi. |

`attack_label` servirà come target. `difficulty` resta metadato: **nessuna delle due colonne entrerà negli input** del futuro modello.

Fonte: [UNB, descrizione dei file e costruzione del difficulty level](https://www.unb.ca/cic/datasets/nsl.html).


## Cosa abbiamo verificato e cosa viene dopo

- Il training è letto senza header, con **41 feature più target e difficulty**.
- Le tre feature testuali richiederanno una trasformazione prima dei modelli.
- Le feature numeriche hanno scale e significati diversi; questo notebook non sceglie ancora preprocessing o feature da eliminare.
- I conteggi effettivi non vengono forzati a coincidere con il paper.

Il controllo qualità è nella sezione 7. Il [notebook 14](14_official_test_preparation.ipynb) descrive la preparazione dei dati per il test ufficiale.


## 7. Controllo qualità prima della modellazione (T04)

Conserviamo tutte le righe. Contiamo mancanti, numeri non finiti, duplicati e colonne costanti.
Per il test ufficiale verifichiamo soltanto struttura, integrità e copertura delle etichette:
nessuna distribuzione guida il modello. I duplicati sono distinti per **43 colonne complete**
e **41 sole feature**; questi ultimi possono avere etichette diverse.

In [ ]:
import numpy as np

official_test = load_nsl_kdd(project_root / 'data/raw/KDDTest+.csv')
audit_rows = []
for name, frame in [('train', train), ('test ufficiale', official_test)]:
    numeric = frame.select_dtypes(include='number')
    audit_rows.append({
        'insieme': name, 'righe': len(frame), 'colonne': frame.shape[1],
        'mancanti': int(frame.isna().sum().sum()),
        'non_finiti': int((~np.isfinite(numeric.to_numpy())).sum()),
        'duplicati_43': int(frame.duplicated().sum()),
        'duplicati_41_feature': int(frame.duplicated(subset=list(FEATURE_COLUMNS)).sum()),
        'costanti': ', '.join(frame.columns[frame.nunique().eq(1)]),
    })
display(pd.DataFrame(audit_rows).set_index('insieme'))

In [ ]:
overlap_rows = {}
for columns in [list(train.columns), list(FEATURE_COLUMNS)]:
    train_keys = pd.MultiIndex.from_frame(train[columns]).unique()
    test_keys = pd.MultiIndex.from_frame(official_test[columns]).unique()
    overlap_rows[f'{len(columns)} colonne'] = len(train_keys.intersection(test_keys))
display(pd.Series(overlap_rows, name='record unici condivisi tra split ufficiali'))
display(train['su_attempted'].value_counts().sort_index().rename('record train'))
display(train['attack_label'].value_counts().rename('record train per attacco'))
print('Etichette presenti solo nel test (controllo di copertura):')
print(sorted(set(official_test.attack_label) - set(train.attack_label)))

### Decisioni di cleaning

**Dati utilizzabili:** nessun valore mancante o non finito; nessuna imputazione necessaria.
`su_attempted` contiene anche 2 (59 righe train): la descrizione UCI lo presenta come indicatore,
ma non documenta questa codifica aggiuntiva. Non trasformiamo arbitrariamente 2 in 0/1.
`num_outbound_cmds` è costante: resta tra le 41 feature per mantenere il protocollo, senza selezione.

Non deduplichiamo: i record completi non sono duplicati; feature identiche possono avere risposte
diverse. L’overlap fra split ufficiali è un limite del benchmark, non un invito a riscrivere il test.
Lo split stratificato interno verrà controllato separatamente nel notebook 02.

I conteggi train/test restano 125.973/22.544, contro 125.972/22.542 del paper: causa non accertata.
Nessuna eliminazione per farli coincidere. Riscontri e limiti in [DATA_AUDIT.md](../docs/DATA_AUDIT.md).
Il prossimo passaggio è definire gli input e l’encoding, poi target/split e solo infine il fit.